# GPIO Example 1 — Digital output, then input bias

**Target:** STM32F103C8T6 · PA4 LED and PB9 button  
**Focus:** Start with a visible output, then configure a stable digital input.

## Feature classification

| Pin role | STM32F1 HAL mode | Variants | Example |
|---|---|---|---|
| Input | `GPIO_MODE_INPUT` | `GPIO_NOPULL`, `GPIO_PULLUP`, `GPIO_PULLDOWN` | PB9 button / sensor logic |
| Output | `GPIO_MODE_OUTPUT_PP` | `GPIO_SPEED_FREQ_LOW`, `MEDIUM`, `HIGH` | PA4 LED through a resistor |
| Output | `GPIO_MODE_OUTPUT_OD` | pull-up required; external or internal | open-drain line / level-compatible sink |
| Analog | `GPIO_MODE_ANALOG` | no digital pull | ADC input |

STM32F1 input, output, and alternate-function modes are configured differently from newer STM32 families. `GPIO_SPEED_FREQ_*` selects output slew/drive behavior, not a software toggle frequency. Start with the lowest speed that meets edge-time needs.

## Example A — Push-pull LED output

The repository's `STM32F103C8T6_leds_rgb_ex1` project uses PA4–PA6 as GPIO outputs. This single-pin version is the first exercise. Set the output latch before switching the pin to output; verify whether the actual LED circuit is active-high or active-low.

```c
GPIO_InitTypeDef gpio = {0};
__HAL_RCC_GPIOA_CLK_ENABLE();
HAL_GPIO_WritePin(GPIOA, GPIO_PIN_4, GPIO_PIN_RESET);
gpio.Pin = GPIO_PIN_4;
gpio.Mode = GPIO_MODE_OUTPUT_PP;
gpio.Pull = GPIO_NOPULL;
gpio.Speed = GPIO_SPEED_FREQ_LOW;
HAL_GPIO_Init(GPIOA, &gpio);
HAL_GPIO_TogglePin(GPIOA, GPIO_PIN_4);
```


## Example B — Pollable PB9 input

The repository's `STM32F103C8T6_PUSH_BUTTON` uses PB9 with no internal pull. That is valid only if the external button circuit provides a defined idle level. This safer standalone wiring example uses an internal pull-up and a normally-open button from PB9 to GND; the pressed state is low. If you choose this wiring, change the CubeMX pin pull to Pull-up.

```c
GPIO_InitTypeDef gpio = {0};
__HAL_RCC_GPIOB_CLK_ENABLE();
gpio.Pin = GPIO_PIN_9;
gpio.Mode = GPIO_MODE_INPUT;
gpio.Pull = GPIO_PULLUP;
HAL_GPIO_Init(GPIOB, &gpio);

GPIO_PinState pressed = (HAL_GPIO_ReadPin(GPIOB, GPIO_PIN_9) == GPIO_PIN_RESET);
```


## Example C — Open-drain output

Open-drain can pull low or release the line; it cannot actively drive high. Add a pull-up to 3.3 V. This is useful for shared low-dominant signals and I2C-style signaling, but I2C pins should normally be configured by the I2C peripheral rather than bit-banged GPIO.

## Behaviour summary

| Configuration | Driven low | Driven high | Typical use |
|---|---|---|---|
| Input + no pull | High impedance | High impedance | External circuit provides both levels |
| Input + pull-up | Reads high when released | Internally biased | Button to ground |
| Output push-pull | Actively low | Actively high | LED/control line |
| Output open-drain | Actively low | Released; pull-up raises line | Shared bus / level translation within electrical limits |

Never connect two push-pull outputs together. Check the datasheet's per-pin and total-port current limits before driving loads.